In [2]:
import numpy as np

# Apply the first 10 images

In [1]:
from tqdm import tqdm
import torch
import torchvision
import torchvision.transforms as transforms
random_seed = 2

transform = transforms.Compose([  
    transforms.ToTensor(),  
    lambda x: (x * 255).to(dtype=torch.uint8) 
])

torch.backends.cudnn.enabled = False
torch.manual_seed(random_seed)

batch_size = 1
random_seed = 1

train_dataset = torchvision.datasets.MNIST(root='D:\\PythonProjects\\JupyterNotebooks\\PixelLevelBPE\\data', train=True, download=True, transform=transform)  # 下载并加载 MNIST 训练数据集，应用上述转换。

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

progress = tqdm(train_loader)

  0%|                                                                                        | 0/60000 [00:00<?, ?it/s]

# Define function

In [3]:
# reshape
def reshape_to_tuples(data, dim):


    data = np.squeeze(np.array(data))
    assert len(data.shape) <= 3, "Data must be equal to or less than 3D"

    if len(data.shape) == 2:
        rows, cols = data.shape
    else:
        raise ValueError("Havn't implemented for 3D data yet")

    row_group_size = rows // dim[0]
    col_group_size = cols // dim[1]

    row_indices = []
    col_indices = []
    for i in range(0, row_group_size):
        row_indices.append([j for j in range(i, rows, row_group_size)])

    for i in range(0, col_group_size):
        col_indices.append([j for j in range(i, cols, col_group_size)])

    tuples = []
    for row_indices_group in row_indices:
        for col_indices_group in col_indices:
            group = []
            for r_idx in row_indices_group:
                for c_idx in col_indices_group:
                    group.append(data[r_idx, c_idx])
            tuples.append(tuple(group))

    return tuples


# get pair frequency
def freq_pair(ids):
    counts = {}  
    for pair in zip(ids, ids[1:]):  
        counts[pair] = counts.get(pair, 0) + 1
    return counts


# get pair with highest frequency
def max_freq_pair(pairs):


    max_freq = None
    for pair, freq in pairs.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq
    return best_pair, max_freq


# merge
def merge(ids, pair, idx):  

    newids = []  
    i = 0  
    while i < len(ids):  
      
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:  
            newids.append(idx)  
            i += 2  
        else:
            newids.append(ids[i])  
            i += 1 
    return newids 

# dim 2*2 tuple list

In [48]:
tuple_list_2_2 = []

for i, (image, label) in enumerate(progress):
    if i == 10:
        break
    tuple_list = reshape_to_tuples(image, dim=(2, 2))
    tuple_list_2_2.append(tuple_list)

tuple_list_2_2 = [item for sublist in tuple_list_2_2 for item in sublist]

## root vocabulary dim 2*2 tuple list

In [49]:
vocab_size = 200
min_freq = 2

root_vocab_2_2 = {}
merges_2_2 = {}
root_code_2_2 = 0
ids = list(tuple_list_2_2)

In [50]:
while len(merges_2_2) < vocab_size:
    stats = freq_pair(ids)
    pair, freq = max_freq_pair(stats)

    if freq < min_freq:
        break

    if pair not in merges_2_2.values():
        idx = len(merges_2_2)
        merges_2_2[idx] = pair
        
        if pair[0] not in root_vocab_2_2.values() and not isinstance(pair[0], int):
            root_vocab_2_2[str(root_code_2_2)] = pair[0]
            root_code_2_2 += 1
        if pair[1] not in root_vocab_2_2.values() and not isinstance(pair[1], int):
            root_vocab_2_2[str(root_code_2_2)] = pair[1]
            root_code_2_2 += 1
    
    ids = merge(ids, pair, idx)

In [60]:
# tuple that will be merged
root_vocab_2_2

keys = list(root_vocab_2_2.keys())
values = list(root_vocab_2_2.values())

for key, value in zip(keys[:5], values[:5]):
    print(f"'{key}': {value}")  # print first 5 rows

'0': (0, 0, 0, 0)
'1': (0, 252, 0, 0)
'2': (0, 0, 252, 0)
'3': (0, 0, 0, 252)
'4': (0, 0, 0, 253)


## tuple_list_2_2 → mapped_tuple_list_2_2
- tuple → code

In [63]:
reverse_root_vocab_2_2 = {v: k for k, v in root_vocab_2_2.items()}  # {tuple: code}

mapped_tuple_list_2_2 = [reverse_root_vocab_2_2.get(item, item) for item in tuple_list_2_2]
print(mapped_tuple_list_2_2[:5])

['47', '4', '4', (0, 0, 0, 119), (0, 0, 0, 25)]


## merge process

In [54]:
num_merges = 50
ids = list(mapped_tuple_list_2_2)

In [55]:
merges_2_2 = {}  
for i in range(num_merges):  
    stats = freq_pair(ids) 
    pair = max(stats, key=stats.get)  
    idx = len(root_vocab_2_2) + i 
    str_idx = str(idx)
    print(f"merging {pair} into a new token '{str_idx}'")
    ids = merge(ids, pair, str_idx) 
    merges_2_2[pair] = str_idx  

merging ('0', '0') into a new token '49'
merging ('49', '49') into a new token '50'
merging ('50', '50') into a new token '51'
merging ('49', '0') into a new token '52'
merging ('51', '0') into a new token '53'
merging ('1', '1') into a new token '54'
merging ('51', '49') into a new token '55'
merging ('2', '2') into a new token '56'
merging ('3', '3') into a new token '57'
merging ('50', '49') into a new token '58'
merging ('51', '52') into a new token '59'
merging ('4', '4') into a new token '60'
merging ('50', '52') into a new token '61'
merging ('5', '5') into a new token '62'
merging ('6', '6') into a new token '63'
merging ('3', '4') into a new token '64'
merging ('7', '7') into a new token '65'
merging ('50', '0') into a new token '66'
merging ('53', '8') into a new token '67'
merging ('67', '9') into a new token '68'
merging ('58', '10') into a new token '69'
merging ('11', '11') into a new token '70'
merging ('4', '3') into a new token '71'
merging ('54', '1') into a new token

## dim 2*2 tuple list vocabulary

In [56]:
for (p0, p1), idx in merges_2_2.items(): 
    root_vocab_2_2[idx] = root_vocab_2_2[p0] + root_vocab_2_2[p1] 

In [59]:
root_vocab_2_2

keys = list(root_vocab_2_2.keys())
values = list(root_vocab_2_2.values())

for key, value in zip(keys[-5:], values[-5:]):
    print(f"'{key}': {value}")  # print last 5 rows

'94': (0, 251, 0, 0, 0, 253, 0, 0, 0, 62, 0, 0)
'95': (0, 0, 64, 0, 0, 0, 251, 0)
'96': (0, 0, 64, 0, 0, 0, 251, 0, 0, 0, 253, 0, 0, 0, 220, 0)
'97': (0, 0, 64, 0, 0, 0, 251, 0, 0, 0, 253, 0, 0, 0, 220, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
'98': (0, 253, 0, 0, 0, 253, 0, 0)


## encode

In [43]:
def encode(mapped_tuple_list, merges):
    tokens = mapped_tuple_list 
    while len(tokens) >= 2:
        stats = freq_pair(tokens)                                 
        pair, freq = max_freq_pair(stats)
        
        if pair not in merges:
            break 
        idx = merges_2_2[pair]
        tokens = merge(tokens, pair, idx)  
       
    return tokens

In [65]:
tuple_list_2_2_encode = encode(mapped_tuple_list_2_2, merges = merges_2_2)
print(tuple_list_2_2_encode[:5])

['47', '60', (0, 0, 0, 119), (0, 0, 0, 25), '51']


### decode

In [45]:
def decode(encoded_list, vocab):
    decoded_list = []

    # decode
    for item in encoded_list:
        if isinstance(item, str) and item in vocab:
            decoded_list.extend(vocab[item])
        else:
            decoded_list.extend(item)
    
    # reshape
    final_list = [tuple(decoded_list[i:i + 4]) for i in range(0, len(decoded_list), 4)]
    
    return final_list

In [68]:
decoded_list_2_2 = decode(tuple_list_2_2_encode, root_vocab_2_2)
print(decoded_list_2_2[:5])

[(0, 0, 0, 240), (0, 0, 0, 253), (0, 0, 0, 253), (0, 0, 0, 119), (0, 0, 0, 25)]


In [47]:
decoded_list_2_2 == tuple_list_2_2

True

# dim 2*1 tuple list